# Emoji provider classification :  improved submission
Predict **apple, facebook, google, messenger, mozilla, samsung, whatsapp** from emoji rendering style.

The initial Extra Trees model scored **0.81838 on Kaggle (reported by the user)** and 82.14% on local validation. The improved model achieves **96.76% on the same 20% validation split**. The new Kaggle score is unknown until submission.

This notebook runs the model that was actually evaluated: color/opacity/texture descriptors + frozen ResNet18 descriptors → standardization → RBF SVM. It does not fine-tune ResNet. The earlier optional fine-tuning workflow is preserved in `ResNet fine-tuning.ipynb`.

Use Python **3.12–3.14**. In Colab, select a GPU runtime for feature extraction. Upload/extract the project code and data together, or put the data in Google Drive. The required code files are `improve.py`, `style_features.py`, `learned_features.py`, `requirements-improved.txt`, and `requirements-baseline.txt`. Keep `train_labels.csv`, `train/`, and `test/` together. An official `sample_submission.csv` is optional and is checked if provided.


In [ ]:
%pip install -q -r requirements-improved.txt


In [ ]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Optional in Colab, if your data are in Drive:
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = Path('.').resolve()  # Or Path('/content/drive/MyDrive/emoji_project')
CODE_DIR = Path('.').resolve()  # Folder containing improve.py and its helper files.
assert (DATA_DIR/'train_labels.csv').exists()
assert (DATA_DIR/'train').is_dir() and (DATA_DIR/'test').is_dir()
assert (CODE_DIR/'improve.py').exists()


## 1. Understand the data
IDs are strings: leading zeros identify the correct files. All PNGs are converted to RGBA, then transparency is composited onto white for the ResNet descriptors. Handcrafted features retain opacity and visible foreground colors. Filenames and PNG encoding metadata are not predictors.


In [ ]:
labels = pd.read_csv(DATA_DIR/'train_labels.csv', dtype={'Id': str})
assert list(labels.columns) == ['Id', 'Label']
assert labels.Id.is_unique and labels.notna().all().all()
display(labels.head())
display(labels.Label.value_counts().rename('count').to_frame())
fig, axes = plt.subplots(1, 7, figsize=(16, 3))
for ax, (name, group) in zip(axes, labels.groupby('Label')):
    with Image.open(DATA_DIR/'train'/f'{group.Id.iloc[0]}.png') as im:
        rgba=im.convert('RGBA')
        rgb=Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba).convert('RGB')
    ax.imshow(rgb); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Model and validation
- **1,607 handcrafted features** summarize foreground colors, color combinations, opacity, gradients, local binary texture patterns, and spatial means/variation.
- **1,920 frozen ResNet18 features** summarize each channel's mean and standard deviation after its four residual stages, at 128×128 input resolution. ImageNet weights require internet on the first run.
- **StandardScaler** is fitted only on the training fold. The SVM uses an RBF kernel, `C=10` and `gamma='scale'`.

The split is 80% training / 20% validation, stratified by provider, seed 42. Development compared three C values for each feature family and two for the combined features. C=10 and C=100 tied for the combined model; the lower C was retained. Model selection reuses validation, so this is a selection estimate rather than an untouched final test score. A second random split is recorded when available; it can share images with the first training split.

The next cell evaluates on validation, refits on **all** labels, saves the model and creates `submission_improved.csv` plus `submission.csv`. Feature caches make subsequent runs faster. The first run takes several minutes or longer on CPU. Frozen features still work without GPU. Check your competition rules permit pretrained ImageNet weights.


In [ ]:
subprocess.run([sys.executable, str(CODE_DIR/'improve.py'),
                '--data-dir', str(DATA_DIR), '--features', 'combined', '--C', '10'], check=True)


## 3. Inspect measured results
These files contain actual measurements from the run above. No test labels are used to train or select the model. Related emoji content can occur across splits; a semantic-group validation split would assess unseen emoji designs more strictly if such group labels become available.


In [ ]:
metrics = json.loads((DATA_DIR/'artifacts/improved_metrics.json').read_text())
print(f"Validation accuracy: {metrics['validation_accuracy']:.2%}")
display(pd.DataFrame(metrics['report']).T)
cm = pd.read_csv(DATA_DIR/'artifacts/improved_confusion_matrix.csv', index_col=0)
fig, ax = plt.subplots(figsize=(8, 6))
image=ax.imshow(cm.to_numpy(), cmap='Blues')
ax.set_xticks(range(len(cm)), cm.columns, rotation=45, ha='right')
ax.set_yticks(range(len(cm)), cm.index)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
for i in range(len(cm)):
    for j in range(len(cm)): ax.text(j, i, str(cm.iloc[i,j]), ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()
second = DATA_DIR/'artifacts/improved_metrics_seed123.json'
if second.exists(): print('Second-split check:', json.loads(second.read_text())['validation_accuracy'])


In [ ]:
errors = pd.read_csv(DATA_DIR/'artifacts/improved_validation_predictions.csv', dtype={'Id': str})
errors = errors[errors.true != errors.predicted]
display(errors.head(10))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax in axes.flat: ax.axis('off')
for ax, (_, row) in zip(axes.flat, errors.head(10).iterrows()):
    with Image.open(DATA_DIR/'train'/f'{row.Id}.png') as im:
        rgba=im.convert('RGBA')
        image=Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba).convert('RGB')
    ax.imshow(image); ax.set_title(f'True: {row.true}\nPredicted: {row.predicted}')
plt.tight_layout(); plt.show()


## 4. Verify and download the submission
Upload **submission_improved.csv** to Kaggle. The identical `submission.csv` is also updated. The original CSV is preserved as `submission_baseline.csv` and, when it existed, `submission_before_improvement.csv`. The new leaderboard score must be measured separately. The official competition template takes precedence over the inferred `Id,Label` format.


In [ ]:
result = pd.read_csv(DATA_DIR/'submission_improved.csv', dtype={'Id': str})
test_ids = {p.stem for p in (DATA_DIR/'test').glob('*.png')}
assert list(result.columns) == ['Id', 'Label']
assert len(result) == len(test_ids) and result.Id.is_unique
assert set(result.Id) == test_ids
assert result.notna().all().all() and set(result.Label) <= set(labels.Label)
print('Verified test predictions:', len(result))
display(result.head())
try:
    from google.colab import files
    files.download(str(DATA_DIR/'submission_improved.csv'))
except ImportError:
    print('Upload:', DATA_DIR/'submission_improved.csv')


## 5. Reproduce, explain, and disclose
`improve.py` also runs independently. Exact packages and model settings are saved in `artifacts/improved_metrics.json`. The fitted scaler and SVM are saved together in `artifacts/improved_model.joblib`; load only trusted model files. Delete the `style_*.npz` and `learned_*.npz` caches if you replace or modify images while keeping the same IDs.

For an optional second-split check, run the following cell. It does not replace the main submission or select new settings. It reuses labeled images from the same dataset and is not an independent test set.


In [ ]:
RUN_SECOND_SPLIT = False
if RUN_SECOND_SPLIT:
    subprocess.run([sys.executable, str(CODE_DIR/'improve.py'), '--data-dir', str(DATA_DIR),
                    '--validation-seed', '123', '--validate-only'], check=True)


This notebook and code were produced with OpenAI Codex assistance. [codebase/AI_USAGE.md](../../../codebase/AI_USAGE.md) records the prompts and scope. The team must understand the feature extraction, standardization, RBF SVM, validation limits, and refitting procedure, run the required individual Colab submissions, and write the report independently. Record your new Kaggle score and actual experiments; do not claim validation accuracy as a Kaggle result.
